# IPIN 2022 classical Android-input replay

## tl;dr

**Bounded pipeline pass, not an accuracy pass.** The frozen B1 replay passed
all parser, 50/100 Hz agreement, callback-batch, magnet-removal, causal, and
gap-response gates on two development User03 sequences and one untouched
validation run over two User05 sequences. The validation sequences contain
no magnetometer rows, so they also exercise the required accelerometer +
gyroscope fallback. No continuous heading or trajectory truth was loaded;
product adoption and a personal pilot therefore remain stopped.

## Context & Methods

The four archive members, user split, estimator, step detector, stride gain,
50/100 Hz rates, and thresholds were preregistered before any selected raw
member was opened. User03 development output and implementation hashes were
committed before User05 validation was range-fetched and evaluated once.

### Key Assumptions

- `ACCE` and `GYRO` are required Android-shaped live inputs; `MAGN` is optional.
- `SensorTimestamp` is the estimator time basis. `AppTimestamp` is only a
  callback-timing proxy and cannot establish Android lifecycle behavior.
- `AHRS`, `POSI`, GNSS route/bearing, radio observations, maps, future samples,
  and completed path shape never enter inference.
- Derived distance and endpoint values test rate self-consistency only; they
  are not errors against truth.

In [1]:
from pathlib import Path
import json

cwd = Path.cwd().resolve()
roots = (cwd / "research" / "pdr", cwd, cwd.parent)
research_root = next(path for path in roots if (path / "pdr_research").is_dir())
manifest_path = research_root / "datasets" / "manifests" / "ipin-classical-result-v1.json"
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))

assert manifest["status"] == "validation-complete"
assert manifest["execution_control"]["validation_run_count"] == 1
assert manifest["execution_control"]["parameter_search_performed"] is False

{
    "experiment": manifest["experiment"],
    "sequences": manifest["aggregate"]["source_sequence_count"],
    "development_user": manifest["execution_control"]["development_user"],
    "untouched_validation_user": manifest["execution_control"]["validation_user"],
    "eligible_sensor_rows": manifest["aggregate"]["eligible_sensor_rows_loaded"],
    "full_archive_downloaded": manifest["acquisition"]["full_archive_downloaded"],
}

{'experiment': 'ipin-2022-classical-replay-v1',
 'sequences': 4,
 'development_user': '03',
 'untouched_validation_user': '05',
 'eligible_sensor_rows': 169280,
 'full_archive_downloaded': False}

## Data

Only aggregate sequence evidence is committed. Raw IPIN members and detailed
replay outputs remain in ignored research data/output directories.

In [2]:
print(f"{'phase':<12} {'sequence':<16} {'rows':>7} {'coverage_s':>11} {'mag':>5} {'steps50':>8} {'steps100':>9}")
print("-" * 76)
for sequence in manifest["sequences"]:
    print(
        f"{sequence['phase']:<12} {sequence['id']:<16} "
        f"{sequence['eligible_rows']:>7} {sequence['common_coverage_s']:>11.3f} "
        f"{str(sequence['magnetometer_present']):>5} "
        f"{sequence['steps_50_hz']:>8} {sequence['steps_100_hz']:>9}"
    )

phase        sequence            rows  coverage_s   mag  steps50  steps100
----------------------------------------------------------------------------
development  trial12-user03     46910     188.958  True      266       264
development  trial13-user03     49824     200.691  True      255       255
validation   trial12-user05     35239     176.689 False      234       232
validation   trial13-user05     37307     187.054 False      242       243


## Results

In [3]:
aggregate = manifest["aggregate"]
{
    "raw_gates_passed": f"{aggregate['raw_gate_pass_count']}/4",
    "replay_gates_passed": f"{aggregate['replay_gate_pass_count']}/4",
    "batch_invariance_checks": f"{aggregate['callback_batch_invariant_count']}/8",
    "gap_stress_checks": f"{aggregate['gap_stress_pass_count']}/8",
    "future_sample_violations": aggregate["future_sample_violations"],
    "validation_sequences_without_magnetometer": aggregate["validation_sequences_without_magnetometer"],
    "max_validation_step_difference_pct": round(100 * aggregate["maximum_validation_step_count_relative_difference"], 3),
    "max_validation_amplitude_difference_pct": round(100 * aggregate["maximum_validation_amplitude_relative_difference"], 3),
    "max_validation_derived_distance_difference_pct": round(100 * aggregate["maximum_validation_distance_relative_difference"], 3),
    "max_validation_endpoint_separation_ratio_pct": round(100 * aggregate["maximum_validation_endpoint_separation_over_longer_distance"], 3),
}

{'raw_gates_passed': '4/4',
 'replay_gates_passed': '4/4',
 'batch_invariance_checks': '8/8',
 'gap_stress_checks': '8/8',
 'future_sample_violations': 0,
 'validation_sequences_without_magnetometer': 2,
 'max_validation_step_difference_pct': 0.855,
 'max_validation_amplitude_difference_pct': 0.808,
 'max_validation_derived_distance_difference_pct': 0.85,
 'max_validation_endpoint_separation_ratio_pct': 1.159}

In [4]:
assert manifest["execution_control"]["user_split_disjoint"] is True
assert manifest["execution_control"]["ground_truth_rows_loaded"] == 0
assert manifest["execution_control"]["platform_ahrs_rows_used"] == 0
assert manifest["execution_control"]["model_weights_loaded"] == 0
assert aggregate["raw_gate_pass_count"] == aggregate["replay_gate_pass_count"] == 4
assert aggregate["callback_batch_invariant_count"] == 8
assert aggregate["magnetometer_removal_invariant_count"] == 8
assert aggregate["gap_stress_pass_count"] == 8
assert aggregate["future_sample_violations"] == 0
assert aggregate["maximum_validation_step_count_relative_difference"] <= 0.02
assert aggregate["maximum_validation_amplitude_relative_difference"] <= 0.03
assert aggregate["maximum_validation_distance_relative_difference"] <= 0.03
assert aggregate["maximum_validation_endpoint_separation_over_longer_distance"] <= 0.05
assert all(value != "PENDING" for value in manifest["implementation_sha256"].values())
assert manifest["accuracy_claim_allowed"] is False
assert manifest["product_adoption_allowed"] is False
assert manifest["personal_pilot_allowed"] is False
print("PASS: bounded pipeline compatibility; accuracy, product adoption, and pilot remain Stop.")

PASS: bounded pipeline compatibility; accuracy, product adoption, and pilot remain Stop.


## Takeaways

- The Android-compatible minimum remains raw accelerometer + gyroscope with
  sensor timestamps at a 50 or 100 Hz processing target. Magnetometer must be
  optional: both untouched User05 sequences omit it and still pass.
- The frozen pipeline is stable across the tested rate, batching, sensor
  absence, and injected-gap transformations on these four IPIN sequences.
- The comparison has no continuous target trajectory or body heading. It
  cannot validate the displayed derived distances or endpoints as accurate.
- Real callback clocks, screen-off collection, foreground-service behavior,
  OEM power policy, battery, thermal behavior, and pocket UX remain outside
  this public-data result.
- Further accuracy work waits for rights-compatible continuous truth or an
  approved multi-user, multi-device capture program; it does not continue as
  another parameter sweep.